# 03. Model Improvement and Tuning

This notebook picks up where the baseline left off.

By now I already know that the data contains useful signal. The next step is to see whether more flexible models can capture that signal better than linear regression without turning the workflow into chaos.

So the mindset here is practical:
- try stronger models that make sense for tabular data
- tune them enough to be fair
- compare them honestly with the baseline
- understand where the best model still struggles


## Why these models?
### Random Forest
Random Forest is a strong next step after Linear Regression because it can capture non-linear patterns and interactions without heavy feature engineering.

### XGBoost
XGBoost is a strong candidate for tabular data because it often handles mixed numeric/categorical patterns very well after encoding, and it supports controlled regularization.

These two models are useful because they improve model flexibility while staying standard and explainable enough for comparison.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


## Load Data and Reuse Baseline Setup
We keep the same target, feature set, and basic preprocessing from the baseline notebook. This keeps the comparison fair.


In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

train = train[train['avg_ctc'].notna()].copy()
test = test[test['avg_ctc'].notna()].copy()

target = 'avg_ctc'
features = ['min_exp', 'max_exp', 'job_title', 'location']

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

numeric_features = ['min_exp', 'max_exp']
categorical_features = ['job_title', 'location']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


## Baseline Reference
We keep the same baseline models from the previous notebook so the improvement is explicit rather than implied.


In [ ]:
baseline_models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Linear regression baseline': LinearRegression(),
}

baseline_results = []
for name, model in baseline_models.items():
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', model),
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    baseline_results.append({
        'model': name,
        'mae': mean_absolute_error(y_test, pred),
        'r2': r2_score(y_test, pred),
    })

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df.assign(mae=baseline_results_df['mae'].round(2), r2=baseline_results_df['r2'].round(4))


## Advanced Model Candidates
Before tuning, we evaluate two stronger models with sensible default settings.

This tells us whether there is meaningful headroom beyond the baseline before we spend effort on hyperparameter search.


In [ ]:
candidate_models = {
    'Random Forest': RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=200,
    ),
    'XGBoost': XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        n_estimators=200,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2,
    ),
}

candidate_results = []
fitted_candidates = {}

for name, model in candidate_models.items():
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', model),
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    fitted_candidates[name] = pipeline
    candidate_results.append({
        'model': name,
        'mae': mean_absolute_error(y_test, pred),
        'r2': r2_score(y_test, pred),
    })

candidate_results_df = pd.DataFrame(candidate_results)
comparison_df = pd.concat([baseline_results_df, candidate_results_df], ignore_index=True).sort_values('mae').reset_index(drop=True)
comparison_df.assign(mae=comparison_df['mae'].round(2), r2=comparison_df['r2'].round(4))


## Interpretation of Untuned Results
At this stage, we want to know whether the advanced models clearly outperform the baseline.

If they do, tuning is justified. If not, tuning is likely not worth the added complexity.


**Observed benchmark from this dataset**
- Mean baseline MAE: about 112,228.65
- Linear regression MAE: about 75,424.52
- Random Forest MAE: about 60,364.23
- XGBoost MAE: about 61,079.56

This shows a clear improvement beyond the baseline, so tuning is justified.


## Hyperparameter Tuning
We use `RandomizedSearchCV` because it gives a strong balance between search quality and runtime.

### Tuning strategy
- scoring metric: negative MAE
- cross-validation: 3-fold
- objective: improve business-readable prediction accuracy without rebuilding the pipeline from scratch


In [ ]:
rf_search = RandomizedSearchCV(
    estimator=Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]),
    param_distributions={
        'model__n_estimators': [200, 300, 500],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 0.7],
    },
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
)

rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
rf_pred = rf_best.predict(X_test)

rf_summary = pd.DataFrame({
    'model': ['Tuned Random Forest'],
    'mae': [mean_absolute_error(y_test, rf_pred)],
    'r2': [r2_score(y_test, rf_pred)],
})

print(rf_search.best_params_)
rf_summary.assign(mae=rf_summary['mae'].round(2), r2=rf_summary['r2'].round(4))


In [ ]:
xgb_search = RandomizedSearchCV(
    estimator=Pipeline([
        ('preprocess', preprocessor),
        ('model', XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)),
    ]),
    param_distributions={
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [4, 5, 6, 8],
        'model__learning_rate': [0.03, 0.05, 0.08, 0.1],
        'model__subsample': [0.8, 0.9, 1.0],
        'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
        'model__reg_alpha': [0, 0.05, 0.1, 0.5],
        'model__reg_lambda': [1, 2, 3, 5],
    },
    n_iter=12,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
)

xgb_search.fit(X_train, y_train)
xgb_best = xgb_search.best_estimator_
xgb_pred = xgb_best.predict(X_test)

xgb_summary = pd.DataFrame({
    'model': ['Tuned XGBoost'],
    'mae': [mean_absolute_error(y_test, xgb_pred)],
    'r2': [r2_score(y_test, xgb_pred)],
})

print(xgb_search.best_params_)
xgb_summary.assign(mae=xgb_summary['mae'].round(2), r2=xgb_summary['r2'].round(4))


## Tuned Model Comparison
This is the key comparison: baseline vs untuned advanced models vs tuned advanced models.


In [ ]:
final_results = pd.concat(
    [comparison_df, rf_summary, xgb_summary],
    ignore_index=True,
).sort_values('mae').reset_index(drop=True)

final_results.assign(mae=final_results['mae'].round(2), r2=final_results['r2'].round(4))


**Observed tuned results from this dataset**
- Tuned Random Forest MAE: about 60,772.17
- Tuned XGBoost MAE: about 60,498.50

The tuned XGBoost model performs best overall on MAE while also matching the top R? performance.


## Error Analysis
A good model is not just the one with the best average metric. We also want to understand where it still fails.

Here we inspect the best model's largest residuals and whether the errors concentrate in certain kinds of rows.


In [ ]:
best_model = xgb_best
best_pred = xgb_pred

error_df = X_test.copy()
error_df['actual_salary'] = y_test.values
error_df['predicted_salary'] = best_pred
error_df['absolute_error'] = (error_df['actual_salary'] - error_df['predicted_salary']).abs()
error_df['residual'] = error_df['actual_salary'] - error_df['predicted_salary']

error_df.sort_values('absolute_error', ascending=False).head(15)


In [ ]:
error_summary = pd.DataFrame({
    'metric': ['Median absolute error', '90th percentile absolute error', 'Max absolute error'],
    'value': [
        error_df['absolute_error'].median(),
        error_df['absolute_error'].quantile(0.9),
        error_df['absolute_error'].max(),
    ],
})

error_by_experience = error_df.groupby(pd.cut(error_df['max_exp'], bins=[-1, 1, 3, 5, 10, 50], labels=['0-1', '2-3', '4-5', '6-10', '10+']), observed=False)['absolute_error'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(error_df['absolute_error'], bins=40, ax=axes[0], color='#4c78a8')
axes[0].set_title('Absolute Error Distribution for Best Model')
axes[0].set_xlabel('Absolute Error')

sns.barplot(data=error_by_experience, x='max_exp', y='absolute_error', ax=axes[1], color='#f58518')
axes[1].set_title('Average Absolute Error by Experience Bucket')
axes[1].set_xlabel('Experience Bucket')
axes[1].set_ylabel('Average Absolute Error')
plt.tight_layout()

error_summary.assign(value=error_summary['value'].round(2)), error_by_experience


### Error Analysis: Interpretation
Typical failure modes in salary data tend to show up in sparse segments, extreme salaries, or rare role-location combinations. If large errors cluster in these regions, that suggests the model is learning the broad market well but struggling in thin-data pockets.


## Feature Importance and Interpretability
For tree-based models, feature importance helps us understand which variables the model relies on most.

This is not a full causal explanation, but it is useful for checking whether the model is paying attention to sensible signals.


In [ ]:
feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
feature_importance = pd.Series(
    best_model.named_steps['model'].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
feature_importance.head(15).sort_values().plot(kind='barh', color='#54a24b')
plt.title('Top 15 Feature Importances for Tuned XGBoost')
plt.xlabel('Importance')
plt.tight_layout()

feature_importance.head(15)


### Interpretability: What to look for
If experience, job title, and location dominate the feature importance ranking, that is directionally consistent with the patterns found in EDA and the baseline notebook. That consistency increases confidence that the model is capturing meaningful structure rather than random noise.


## Final Conclusion
This notebook shows a pretty clear pattern.

The biggest jump comes from moving beyond the linear baseline to tree-based models. Hyperparameter tuning helps, but it is more of a refinement than the main story.

The tuned XGBoost model ends up being the best fit for this project because it improves the error meaningfully while still behaving well enough to explain and deploy.

That makes it a good final model choice, even if there is still room to improve the data and the edge cases later.
